# Arabic Sentiment Analysis — DistilAraBERT (Built From Scratch)
Model  : aubmindlab/distilbert-base-arabic




Reference:

 https://arxiv.org/abs/1910.01108 (DistilBERT)

https://arxiv.org/abs/2003.00104

In [1]:


import os
import json
import math
import warnings
import random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

# CONFIGURATION
TRAIN_PATH    = "/content/train.csv"
VAL_PATH      = "/content/val.csv"
TEST_PATH     = "/content/test.csv"
TEXT_COL       = "clean_text"
LABEL_COL      = "labels"

HF_MODEL_ID = "distilbert-base-multilingual-cased"
NUM_LABELS     = 2       # binary: positive / negative
MAX_SEQ_LEN    = 128     # maximum token length (128 is efficient for tweets)
BATCH_SIZE     = 32
EPOCHS         = 5
LR             = 2e-5    # standard fine-tuning LR for BERT-family
WARMUP_RATIO   = 0.1     # 10% of total steps used for LR warmup
WEIGHT_DECAY   = 0.01
GRAD_CLIP      = 1.0     # gradient clipping for training stability
DROPOUT_CLS    = 0.2     # dropout on the classification head

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device : {DEVICE}")


✅ Device : cuda


In [2]:
print("\n" + "=" * 65)
print("  📦  Downloading DistilAraBERT weights (raw files)")
print("=" * 65)

from huggingface_hub import hf_hub_download

# Download model config (architecture hyperparameters)
config_path = hf_hub_download(repo_id=HF_MODEL_ID, filename="config.json")
with open(config_path) as f:
    hf_config = json.load(f)

print(f"  Config loaded from : {config_path}")
print(f"  Model type         : {hf_config.get('model_type', 'distilbert')}")
print(f"  Hidden size        : {hf_config.get('dim', 768)}")
print(f"  Num layers         : {hf_config.get('n_layers', 6)}")
print(f"  Num heads          : {hf_config.get('n_heads', 12)}")
print(f"  Vocab size         : {hf_config.get('vocab_size', 64000)}\n")

# Try safetensors first (newer format), fall back to pytorch_model.bin
try:
    weights_path = hf_hub_download(repo_id=HF_MODEL_ID, filename="model.safetensors")
    USE_SAFETENSORS = True
    print(f"  Weights format     : safetensors")
except Exception:
    weights_path = hf_hub_download(repo_id=HF_MODEL_ID, filename="pytorch_model.bin")
    USE_SAFETENSORS = False
    print(f"  Weights format     : pytorch (.bin)")

print(f"  Weights path       : {weights_path}")

# Load raw state dict
if USE_SAFETENSORS:
    from safetensors.torch import load_file as safetensors_load
    raw_state_dict = safetensors_load(weights_path, device="cpu")
else:
    raw_state_dict = torch.load(weights_path, map_location="cpu")

print(f"\n  Pre-trained keys   : {len(raw_state_dict):,} tensors loaded ✓")



  📦  Downloading DistilAraBERT weights (raw files)


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

  Config loaded from : /root/.cache/huggingface/hub/models--distilbert-base-multilingual-cased/snapshots/45c032ab32cc946ad88a166f7cb282f58c753c2e/config.json
  Model type         : distilbert
  Hidden size        : 768
  Num layers         : 6
  Num heads          : 12
  Vocab size         : 119547



model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

  Weights format     : safetensors
  Weights path       : /root/.cache/huggingface/hub/models--distilbert-base-multilingual-cased/snapshots/45c032ab32cc946ad88a166f7cb282f58c753c2e/model.safetensors

  Pre-trained keys   : 105 tensors loaded ✓


In [3]:

VOCAB_SIZE         = hf_config.get("vocab_size",         64000)
HIDDEN_DIM         = hf_config.get("dim",                768)
NUM_LAYERS         = hf_config.get("n_layers",           6)
NUM_HEADS          = hf_config.get("n_heads",            12)
FFN_DIM            = hf_config.get("hidden_dim",         3072)
MAX_POSITION       = hf_config.get("max_position_embeddings", 512)
ATTN_DROPOUT       = hf_config.get("attention_dropout",  0.1)
HIDDEN_DROPOUT     = hf_config.get("dropout",            0.1)
HEAD_DIM           = HIDDEN_DIM // NUM_HEADS   # dimension per attention head


def gelu(x: torch.Tensor) -> torch.Tensor:
    """Gaussian Error Linear Unit — exact formulation used in BERT/DistilBERT."""
    return 0.5 * x * (1.0 + torch.tanh(
        math.sqrt(2.0 / math.pi) * (x + 0.044715 * x.pow(3))
    ))


class MultiHeadSelfAttention(nn.Module):

    def __init__(self):
        super().__init__()
        assert HIDDEN_DIM % NUM_HEADS == 0, "hidden_dim must be divisible by num_heads"

        self.n_heads  = NUM_HEADS
        self.head_dim = HEAD_DIM
        self.scale    = math.sqrt(self.head_dim)

        # Separate linear projections (matches HF DistilBERT naming)
        self.q_lin   = nn.Linear(HIDDEN_DIM, HIDDEN_DIM)
        self.k_lin   = nn.Linear(HIDDEN_DIM, HIDDEN_DIM)
        self.v_lin   = nn.Linear(HIDDEN_DIM, HIDDEN_DIM)
        self.out_lin = nn.Linear(HIDDEN_DIM, HIDDEN_DIM)

        self.attn_drop = nn.Dropout(ATTN_DROPOUT)

    def _split_heads(self, x: torch.Tensor, batch: int) -> torch.Tensor:
        x = x.view(batch, -1, self.n_heads, self.head_dim)
        return x.permute(0, 2, 1, 3)

    def _merge_heads(self, x: torch.Tensor, batch: int) -> torch.Tensor:
        x = x.permute(0, 2, 1, 3).contiguous()
        return x.view(batch, -1, self.n_heads * self.head_dim)

    def forward(
        self,
        query: torch.Tensor,           # (B, T, D)
        key: torch.Tensor,             # (B, T, D)
        value: torch.Tensor,           # (B, T, D)
        attention_mask: torch.Tensor,  # (B, 1, 1, T) — additive mask
    ) -> torch.Tensor:
        batch = query.size(0)

        Q = self._split_heads(self.q_lin(query), batch)   # (B, H, T, d)
        K = self._split_heads(self.k_lin(key),   batch)
        V = self._split_heads(self.v_lin(value), batch)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # (B, H, T, T)
        scores = scores + attention_mask                              # mask PAD tokens
        weights = F.softmax(scores, dim=-1)
        weights = self.attn_drop(weights)

        context = torch.matmul(weights, V)                           # (B, H, T, d)
        context = self._merge_heads(context, batch)                  # (B, T, D)
        return self.out_lin(context)


class FeedForwardNetwork(nn.Module):

    def __init__(self):
        super().__init__()
        self.lin1 = nn.Linear(HIDDEN_DIM, FFN_DIM)
        self.lin2 = nn.Linear(FFN_DIM,    HIDDEN_DIM)
        self.drop = nn.Dropout(HIDDEN_DROPOUT)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.lin2(self.drop(gelu(self.lin1(x))))


class TransformerBlock(nn.Module):

    def __init__(self):
        super().__init__()
        self.attention       = MultiHeadSelfAttention()
        self.sa_layer_norm   = nn.LayerNorm(HIDDEN_DIM, eps=1e-12)
        self.ffn             = FeedForwardNetwork()
        self.output_layer_norm = nn.LayerNorm(HIDDEN_DIM, eps=1e-12)
        self.drop            = nn.Dropout(HIDDEN_DROPOUT)

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor) -> torch.Tensor:
        # Self-attention sub-layer (pre-norm is standard in DistilBERT)
        attn_out = self.attention(x, x, x, attn_mask)
        x = self.sa_layer_norm(x + self.drop(attn_out))

        # Feed-forward sub-layer
        ffn_out = self.ffn(x)
        x = self.output_layer_norm(x + self.drop(ffn_out))
        return x


class DistilBERTEmbeddings(nn.Module):

    def __init__(self):
        super().__init__()
        self.word_embeddings     = nn.Embedding(VOCAB_SIZE, HIDDEN_DIM, padding_idx=0)
        self.position_embeddings = nn.Embedding(MAX_POSITION, HIDDEN_DIM)
        self.LayerNorm           = nn.LayerNorm(HIDDEN_DIM, eps=1e-12)
        self.dropout             = nn.Dropout(HIDDEN_DROPOUT)

        # Pre-register position ids buffer (not a trained parameter)
        self.register_buffer(
            "position_ids",
            torch.arange(MAX_POSITION).unsqueeze(0),   # (1, max_pos)
        )

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        seq_len   = input_ids.size(1)
        pos_ids   = self.position_ids[:, :seq_len]

        word_emb  = self.word_embeddings(input_ids)    # (B, T, D)
        pos_emb   = self.position_embeddings(pos_ids)  # (1, T, D)

        embeddings = word_emb + pos_emb
        embeddings = self.LayerNorm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings


class DistilBERTEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.ModuleList([TransformerBlock() for _ in range(NUM_LAYERS)])

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor) -> torch.Tensor:
        for block in self.layer:
            x = block(x, attn_mask)
        return x


class DistilBERT(nn.Module):

    def __init__(self):
        super().__init__()
        self.embeddings  = DistilBERTEmbeddings()
        self.transformer = DistilBERTEncoder()

    def forward(
        self,
        input_ids:      torch.Tensor,   # (B, T)
        attention_mask: torch.Tensor,   # (B, T) — 1=real token, 0=padding
    ) -> torch.Tensor:
        # Convert 2-D mask → 4-D additive mask for attention scores
        # shape: (B, 1, 1, T)  values: 0.0 (attend) or -10000.0 (ignore)
        ext_mask = (1.0 - attention_mask[:, None, None, :].float()) * -10000.0

        hidden = self.embeddings(input_ids)
        hidden = self.transformer(hidden, ext_mask)
        return hidden   # (B, T, D)


class DistilAraBERTForClassification(nn.Module):

    def __init__(self, num_labels: int = NUM_LABELS):
        super().__init__()
        self.distilbert = DistilBERT()
        # Classification head (pre_classifier + classifier in HF naming)
        self.pre_classifier = nn.Linear(HIDDEN_DIM, HIDDEN_DIM)
        self.classifier     = nn.Linear(HIDDEN_DIM, num_labels)
        self.dropout        = nn.Dropout(DROPOUT_CLS)

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        hidden = self.distilbert(input_ids, attention_mask)  # (B, T, D)
        cls_token = hidden[:, 0, :]                          # (B, D) — [CLS]
        pooled    = gelu(self.pre_classifier(cls_token))     # (B, D)
        pooled    = self.dropout(pooled)
        logits    = self.classifier(pooled)                  # (B, num_labels)
        return logits



In [4]:
print("\n" + "=" * 65)
print("  🔄  Mapping and loading pre-trained DistilAraBERT weights")
print("=" * 65)

def map_hf_weights_to_custom(
    hf_state: dict,
    model: DistilAraBERTForClassification,
) -> dict:

    our_state = model.state_dict()
    mapped    = {}
    skipped   = []
    missing   = []

    PREFIX_MAP = {
        "distilbert.embeddings.":             "distilbert.embeddings.",
        "distilbert.transformer.layer.":      "distilbert.transformer.layer.",
    }

    for hf_key, hf_tensor in hf_state.items():
        our_key = hf_key

        if our_key in our_state:
            if our_state[our_key].shape == hf_tensor.shape:
                mapped[our_key] = hf_tensor
            else:
                skipped.append(
                    f"  ⚠️  Shape mismatch : {our_key} "
                    f"ours={tuple(our_state[our_key].shape)} "
                    f"hf={tuple(hf_tensor.shape)}"
                )
        else:
            missing.append(hf_key)

    # Report mapping statistics
    n_matched  = len(mapped)
    n_total_hf = len(hf_state)
    n_our      = len(our_state)
    print(f"  HF checkpoint tensors   : {n_total_hf:,}")
    print(f"  Our model tensors       : {n_our:,}")
    print(f"  Successfully mapped     : {n_matched:,}")
    print(f"  Random-init (head/new)  : {n_our - n_matched:,}")
    if skipped:
        for msg in skipped:
            print(msg)
    if missing:
        print(f"  HF keys not in our model: {len(missing)}")

    return mapped


model = DistilAraBERTForClassification(num_labels=NUM_LABELS)

mapped_weights = map_hf_weights_to_custom(raw_state_dict, model)

load_result = model.load_state_dict(mapped_weights, strict=False)
print(f"\n  Missing keys (random-init) : {load_result.missing_keys}")
print(f"  ✅  Pre-trained weights loaded successfully\n")

model = model.to(DEVICE)



  🔄  Mapping and loading pre-trained DistilAraBERT weights
  HF checkpoint tensors   : 105
  Our model tensors       : 105
  Successfully mapped     : 100
  Random-init (head/new)  : 5
  HF keys not in our model: 5

  Missing keys (random-init) : ['distilbert.embeddings.position_ids', 'pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
  ✅  Pre-trained weights loaded successfully



In [5]:
print("=" * 65)
print("  🔤  Loading AraBERT WordPiece Tokenizer (utility only)")
print("=" * 65)

from transformers import AutoTokenizer   # tokenizer only — no model loaded
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
print(f"  Vocab size         : {tokenizer.vocab_size:,}")
print(f"  Max model length   : {tokenizer.model_max_length}")
print(f"  [CLS] token id     : {tokenizer.cls_token_id}")
print(f"  [SEP] token id     : {tokenizer.sep_token_id}")
print(f"  [PAD] token id     : {tokenizer.pad_token_id}\n")


  🔤  Loading AraBERT WordPiece Tokenizer (utility only)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  Vocab size         : 119,547
  Max model length   : 512
  [CLS] token id     : 101
  [SEP] token id     : 102
  [PAD] token id     : 0



In [6]:
class ArabicSentimentDataset(Dataset):

    def __init__(self, df: pd.DataFrame, max_len: int = MAX_SEQ_LEN):
        self.texts    = df[TEXT_COL].fillna("").astype(str).tolist()
        self.labels   = df[LABEL_COL].astype(int).tolist()
        self.max_len  = max_len

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict:
        text = self.texts[idx]

        # WordPiece tokenisation with truncation and padding
        encoding = tokenizer(
            text,
            max_length       = self.max_len,
            padding          = "max_length",    # pad to max_len with [PAD]
            truncation       = True,            # truncate longer sequences
            return_tensors   = "pt",            # return PyTorch tensors
            return_token_type_ids = False,      # DistilBERT has no segment ids
        )

        return {
            "input_ids":      encoding["input_ids"].squeeze(0),       # (T,)
            "attention_mask": encoding["attention_mask"].squeeze(0),  # (T,)
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }


def make_loader(df: pd.DataFrame, shuffle: bool, batch_size: int = BATCH_SIZE) -> DataLoader:
    dataset = ArabicSentimentDataset(df)
    return DataLoader(
        dataset,
        batch_size  = batch_size,
        shuffle     = shuffle,
        num_workers = 0,       # safe default; increase for faster I/O
        pin_memory  = DEVICE.type == "cuda",
    )



In [7]:
print("=" * 65)
print("  📂  Loading Data")
print("=" * 65)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"  Train : {len(train_df):,} rows")
print(f"  Val   : {len(val_df):,}   rows")
print(f"  Test  : {len(test_df):,}  rows")
print(f"\n  Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# Verify columns exist
for col in [TEXT_COL, LABEL_COL]:
    assert col in train_df.columns, f"Column '{col}' not found in train CSV"

train_loader = make_loader(train_df, shuffle=True)
val_loader   = make_loader(val_df,   shuffle=False)
test_loader  = make_loader(test_df,  shuffle=False)

  📂  Loading Data
  Train : 5,068 rows
  Val   : 1,086   rows
  Test  : 1,086  rows

  Label distribution (train):
labels
1    2576
0    2492
Name: count, dtype: int64



In [8]:
backbone_params = [p for n, p in model.named_parameters() if "distilbert" in n]
head_params     = [p for n, p in model.named_parameters() if "distilbert" not in n]

optimizer = AdamW(
    [
        {"params": backbone_params, "lr": LR,     "weight_decay": WEIGHT_DECAY},
        {"params": head_params,     "lr": LR * 5, "weight_decay": WEIGHT_DECAY},
    ],
    eps   = 1e-8,
    betas = (0.9, 0.999),
)

total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)

def get_lr_lambda(current_step: int) -> float:
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return max(0.0, 1.0 - progress)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr_lambda)

# Class-balanced loss weights (handles label imbalance)
label_counts = train_df[LABEL_COL].value_counts().sort_index()
total_count  = len(train_df)
class_weights = torch.tensor(
    [total_count / (NUM_LABELS * c) for c in label_counts.values],
    dtype=torch.float
).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print(f"  Total training steps : {total_steps:,}")
print(f"  Warmup steps         : {warmup_steps:,}")
print(f"  Class weights        : {class_weights.cpu().tolist()}\n")


  Total training steps : 795
  Warmup steps         : 79
  Class weights        : [1.016853928565979, 0.9836956262588501]



In [9]:
def train_one_epoch(model, loader, optimizer, scheduler, criterion, epoch):
    model.train()
    total_loss  = 0.0
    all_preds   = []
    all_labels  = []

    for step, batch in enumerate(loader):
        input_ids  = batch["input_ids"].to(DEVICE)
        attn_mask  = batch["attention_mask"].to(DEVICE)
        labels     = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attn_mask)       # (B, num_labels)
        loss   = criterion(logits, labels)
        loss.backward()

        # Gradient clipping prevents exploding gradients during fine-tuning
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

        if (step + 1) % 50 == 0:
            lr_now = scheduler.get_last_lr()[0]
            print(f"    Epoch {epoch+1} | Step {step+1:4d}/{len(loader)} "
                  f"| Loss {loss.item():.4f} | LR {lr_now:.2e}")

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, acc, f1


@torch.no_grad()
def evaluate_loader(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds  = []
    all_labels = []

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attn_mask = batch["attention_mask"].to(DEVICE)
        labels    = batch["label"].to(DEVICE)

        logits     = model(input_ids, attn_mask)
        loss       = criterion(logits, labels)
        total_loss += loss.item()

        preds = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    return avg_loss, np.array(all_preds), np.array(all_labels)


def print_metrics(name, y_true, y_pred, label_names=None):
    print(f"\n{'─'*65}")
    print(f"  {name}")
    print(f"{'─'*65}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")



In [10]:
print("=" * 65)
print("  🚀  Fine-tuning DistilAraBERT")
print("=" * 65)

label_names = ["Negative", "Positive"]
best_val_f1 = 0.0
best_model_state = None
history = []

for epoch in range(EPOCHS):
    print(f"\n{'='*65}")
    print(f"  EPOCH {epoch + 1} / {EPOCHS}")
    print(f"{'='*65}")

    # ── Train ────────────────────────────────────────────────────────────────
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, optimizer, scheduler, criterion, epoch
    )

    # ── Validate ─────────────────────────────────────────────────────────────
    val_loss, val_preds, val_labels = evaluate_loader(model, val_loader, criterion)
    val_acc = accuracy_score(val_labels, val_preds)
    val_f1  = f1_score(val_labels, val_preds, average="macro", zero_division=0)

    history.append({
        "epoch":      epoch + 1,
        "train_loss": train_loss,
        "train_acc":  train_acc,
        "train_f1":   train_f1,
        "val_loss":   val_loss,
        "val_acc":    val_acc,
        "val_f1":     val_f1,
    })

    print(f"\n  ── Epoch {epoch+1} Summary ──────────────────────────────────")
    print(f"  Train Loss : {train_loss:.4f} | Train Acc : {train_acc:.4f} | Train F1 : {train_f1:.4f}")
    print(f"  Val   Loss : {val_loss:.4f} | Val   Acc : {val_acc:.4f} | Val   F1 : {val_f1:.4f}")

    # ── Save best checkpoint ──────────────────────────────────────────────────
    if val_f1 > best_val_f1:
        best_val_f1    = val_f1
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        torch.save(best_model_state, "best_distilArabert.pt")
        print(f"  ✅ New best model saved (val F1 = {best_val_f1:.4f})")

print("\n" + "=" * 65)
print("  📊  Final Evaluation (best checkpoint)")
print("=" * 65)

# Reload best weights
model.load_state_dict(best_model_state)
model.eval()

# Validation
_, val_preds, val_labels = evaluate_loader(model, val_loader, criterion)
print_metrics("Validation Set", val_labels, val_preds, label_names)

# Test
_, test_preds, test_labels = evaluate_loader(model, test_loader, criterion)
print_metrics("Test Set", test_labels, test_preds, label_names)


print("\n" + "=" * 65)
print("  📈  Training History")
print("=" * 65)
history_df = pd.DataFrame(history)
print(history_df.to_string(index=False, float_format="{:.4f}".format))
print("\n  Model saved to: best_distilArabert.pt")

  🚀  Fine-tuning DistilAraBERT

  EPOCH 1 / 5
    Epoch 1 | Step   50/159 | Loss 0.6232 | LR 1.27e-05
    Epoch 1 | Step  100/159 | Loss 0.5261 | LR 1.94e-05
    Epoch 1 | Step  150/159 | Loss 0.4012 | LR 1.80e-05

  ── Epoch 1 Summary ──────────────────────────────────
  Train Loss : 0.6063 | Train Acc : 0.6529 | Train F1 : 0.6529
  Val   Loss : 0.4937 | Val   Acc : 0.7661 | Val   F1 : 0.7619
  ✅ New best model saved (val F1 = 0.7619)

  EPOCH 2 / 5
    Epoch 2 | Step   50/159 | Loss 0.3247 | LR 1.64e-05
    Epoch 2 | Step  100/159 | Loss 0.3679 | LR 1.50e-05
    Epoch 2 | Step  150/159 | Loss 0.4506 | LR 1.36e-05

  ── Epoch 2 Summary ──────────────────────────────────
  Train Loss : 0.4380 | Train Acc : 0.7964 | Train F1 : 0.7962
  Val   Loss : 0.4557 | Val   Acc : 0.7864 | Val   F1 : 0.7863
  ✅ New best model saved (val F1 = 0.7863)

  EPOCH 3 / 5
    Epoch 3 | Step   50/159 | Loss 0.3204 | LR 1.19e-05
    Epoch 3 | Step  100/159 | Loss 0.5687 | LR 1.05e-05
    Epoch 3 | Step  150/

In [11]:
print("=" * 65)
print("  🚀  Fine-tuning DistilAraBERT (Optimized)")
print("=" * 65)

label_names = ["Negative", "Positive"]

best_val_f1 = 0.0
best_model_state = None
history = []

# ✅ Early stopping config
patience = 2
no_improve = 0

for epoch in range(EPOCHS):
    print(f"\n{'='*65}")
    print(f"  EPOCH {epoch + 1} / {EPOCHS}")
    print(f"{'='*65}")

    # ── Train ─────────────────────────────────────────────
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, optimizer, scheduler, criterion, epoch
    )

    # ── Validate ──────────────────────────────────────────
    val_loss, val_preds, val_labels = evaluate_loader(model, val_loader, criterion)
    val_acc = accuracy_score(val_labels, val_preds)
    val_f1  = f1_score(val_labels, val_preds, average="macro", zero_division=0)

    history.append({
        "epoch":      epoch + 1,
        "train_loss": train_loss,
        "train_acc":  train_acc,
        "train_f1":   train_f1,
        "val_loss":   val_loss,
        "val_acc":    val_acc,
        "val_f1":     val_f1,
    })

    print(f"\n  ── Epoch {epoch+1} Summary ───────────────────────────────")
    print(f"  Train Loss : {train_loss:.4f} | Train Acc : {train_acc:.4f} | Train F1 : {train_f1:.4f}")
    print(f"  Val   Loss : {val_loss:.4f} | Val   Acc : {val_acc:.4f} | Val   F1 : {val_f1:.4f}")

    # ── Save best checkpoint ──────────────────────────────
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        torch.save(best_model_state, "best_distilArabert.pt")

        no_improve = 0  # ✅ reset counter

        print(f"  ✅ New best model saved (val F1 = {best_val_f1:.4f})")

    else:
        no_improve += 1
        print(f"  ⚠️ No improvement ({no_improve}/{patience})")

    # ── Early stopping ────────────────────────────────────
    if no_improve >= patience:
        print("\n⛔ Early stopping triggered — stopping training.")
        break


# ─────────────────────────────────────────────────────────
# FINAL EVALUATION
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  📊  Final Evaluation (best checkpoint)")
print("=" * 65)

# Safety check
if best_model_state is None:
    raise ValueError("No model was saved. Check training loop.")

# Load best model
model.load_state_dict(best_model_state)
model.eval()

# Validation
_, val_preds, val_labels = evaluate_loader(model, val_loader, criterion)
print_metrics("Validation Set", val_labels, val_preds, label_names)

# Test
_, test_preds, test_labels = evaluate_loader(model, test_loader, criterion)
print_metrics("Test Set", test_labels, test_preds, label_names)


# ─────────────────────────────────────────────────────────
# TRAINING HISTORY
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  📈  Training History")
print("=" * 65)

history_df = pd.DataFrame(history)
print(history_df.to_string(index=False, float_format="{:.4f}".format))

# Best epoch info
best_epoch = history_df.loc[history_df["val_f1"].idxmax()]
print("\n🏆 Best Epoch Summary:")
print(best_epoch.to_string(float_format="{:.4f}".format))

print("\n  Model saved to: best_distilArabert.pt")

  🚀  Fine-tuning DistilAraBERT (Optimized)

  EPOCH 1 / 5
    Epoch 1 | Step   50/159 | Loss 0.3024 | LR 0.00e+00
    Epoch 1 | Step  100/159 | Loss 0.2138 | LR 0.00e+00
    Epoch 1 | Step  150/159 | Loss 0.3181 | LR 0.00e+00

  ── Epoch 1 Summary ───────────────────────────────
  Train Loss : 0.2485 | Train Acc : 0.9027 | Train F1 : 0.9026
  Val   Loss : 0.4523 | Val   Acc : 0.8177 | Val   F1 : 0.8174
  ✅ New best model saved (val F1 = 0.8174)

  EPOCH 2 / 5
    Epoch 2 | Step   50/159 | Loss 0.1613 | LR 0.00e+00
    Epoch 2 | Step  100/159 | Loss 0.2345 | LR 0.00e+00
    Epoch 2 | Step  150/159 | Loss 0.4978 | LR 0.00e+00

  ── Epoch 2 Summary ───────────────────────────────
  Train Loss : 0.2477 | Train Acc : 0.8964 | Train F1 : 0.8963
  Val   Loss : 0.4523 | Val   Acc : 0.8177 | Val   F1 : 0.8174
  ⚠️ No improvement (1/2)

  EPOCH 3 / 5
    Epoch 3 | Step   50/159 | Loss 0.2180 | LR 0.00e+00
    Epoch 3 | Step  100/159 | Loss 0.3568 | LR 0.00e+00
    Epoch 3 | Step  150/159 | Loss 